In [7]:
# dataset
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

# utils
import torch

# my code
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME

# set up env secrets
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env", override=True)

False

In [8]:
def validate_dataset(ds, batch_size=1, num_workers=0):
    from torch.utils.data import DataLoader

    test_dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    it = iter(test_dl)

    errors = []
    for idx in range(len(ds)):
        try:
            _ = next(it)
        except Exception as e:
            errors.append((idx, repr(e)))

    if errors:
        print(f"❌ Found {len(errors)} invalid samples")
        for idx, err in errors:
            print(f"  - index {idx}: {err}")
    else:
        print("✅ Dataset fully valid")

    return errors


In [9]:
def map_bad_indices_to_episodes(bad_indices, ds_meta):
    # Build start/end index ranges for each episode
    starts, ends = {}, {}
    curr = 0
    for ep_id in ds_meta.episodes:   # keys, e.g. 0, 1, 2...
        L = ds_meta.episodes[ep_id]["length"]
        starts[ep_id] = curr
        ends[ep_id] = curr + L - 1
        curr += L

    # Map bad indices to episodes
    bad_episodes = {}
    for idx in bad_indices:
        for ep_id in ds_meta.episodes:
            if starts[ep_id] <= idx <= ends[ep_id]:
                if ep_id not in bad_episodes:
                    bad_episodes[ep_id] = []
                # store frame index relative to episode
                bad_episodes[ep_id].append(idx - starts[ep_id])
                break

    return bad_episodes


In [10]:
REPO_NAME = 'so101_leg_pick_and_place'
EXPERIMENT_NAME = '50_episodes_v0'
DATASET_PATH = DATASETS_DIR / REPO_NAME
DATASET_ID = f"{HF_NAME}/{REPO_NAME}"

In [11]:
# select episodes
ds_meta= LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)
bad_eps = {10, 11, 22}
episodes = [ep for ep in range(len(ds_meta.episodes)) if ep not in bad_eps]
ds = LeRobotDataset(f"{HF_NAME}/{REPO_NAME}", root=f"{DATASETS_DIR}\\{REPO_NAME}") # episodes=episodes,

In [12]:
errors = validate_dataset(ds, batch_size=1, num_workers=0)

if errors:
    # mitigation: drop bad samples
    bad_indices = [i for i, _ in errors]
    print("Bad indices:", bad_indices)
    # Option 1: filter dataset
    ds = torch.utils.data.Subset(ds, [i for i in range(len(ds)) if i not in bad_indices])
else:
    print("Dataset ready for training")


C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\torchvision\io\_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


✅ Dataset fully valid
Dataset ready for training


In [ ]:
bad_episodes = map_bad_indices_to_episodes(bad_indices, ds_meta)
for ep_id, frames in bad_episodes.items():
    print(f"Episode {ep_id} bad frames: {min(frames)}–{max(frames)} "
          f"(total {len(frames)} frames)")